# Memory Profiling / 内存监控分析

Reads `logs/stage{N}/{run_id}/memory.csv` (produced by `src.monitoring.memory_watcher`) and plots the VRAM/RAM usage over wall-clock time.

读取 `MemoryWatcher` 生成的 CSV 文件，画显存/内存随时间曲线；用于验证 Axiom 5 (Fragmentation Governance) 是否达标（slope ≤ 0.2 GB/h）。

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.platform import logs_dir

In [ ]:
# ---- Configure which run to inspect ----
STAGE = 0
RUN_ID = None   # None ⇒ latest

stage_dir = logs_dir() / f'stage{STAGE}'
if RUN_ID is None:
    runs = sorted([p for p in stage_dir.glob('*') if p.is_dir()])
    if not runs:
        raise FileNotFoundError(f'no runs under {stage_dir}')
    run_dir = runs[-1]
else:
    run_dir = stage_dir / RUN_ID

csv_path = run_dir / 'memory.csv'
print('Loading', csv_path)
df = pd.read_csv(csv_path)
df['ts_dt'] = pd.to_datetime(df['ts'], unit='s')
df['t_hours'] = (df['ts'] - df['ts'].iloc[0]) / 3600
df['used_gb'] = df['used_bytes'] / 1024**3
df['rss_gb'] = df['process_rss_bytes'] / 1024**3
df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(df['t_hours'], df['used_gb'], label=f'{df.kind.iloc[0]} used')
axes[0].set_ylabel('memory used (GB)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(df['t_hours'], df['rss_gb'], color='C1', label='process RSS')
axes[1].set_xlabel('t (hours)')
axes[1].set_ylabel('process RSS (GB)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle(f'Memory · Stage {STAGE} · {run_dir.name}')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Compute VRAM slope over rolling 1h window ----
if len(df) >= 2:
    total_dt_h = df['t_hours'].iloc[-1] - df['t_hours'].iloc[0]
    total_dgb = df['used_gb'].iloc[-1] - df['used_gb'].iloc[0]
    if total_dt_h > 0:
        slope = total_dgb / total_dt_h
        print(f'End-to-end used_gb slope: {slope:+.4f} GB/h over {total_dt_h:.2f}h')
        threshold = 0.2
        status = 'PASS' if abs(slope) <= threshold else 'FAIL'
        print(f'Slope threshold {threshold} GB/h: {status}')
    else:
        print('Not enough time for slope estimate.')
else:
    print('Not enough samples.')